In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split

df = pd.read_csv("breast_cancer.csv")

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify column types using the feature data
numeric_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Preprocessing for numeric columns
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the learned transformations to test data
X_test_processed = preprocessor.transform(X_test)

print("Numeric columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print("Training contains NaN:", pd.isna(X_train_processed).any())
print("Testing contains NaN:", pd.isna(X_test_processed).any())

Numeric columns: 30
Categorical columns: 0
Processed training shape: (455, 30)
Processed testing shape: (114, 30)
Training contains NaN: False
Testing contains NaN: False


In [6]:
# Verify that the scaler was fitted on the training rows
scaler = preprocessor.named_transformers_["numeric"].named_steps["scaler"]

print("Scaler training sample count:", scaler.n_samples_seen_)
print("Expected training sample count:", len(X_train))

assert scaler.n_samples_seen_ == len(X_train)

# Verify that train and test rows remain separate
assert X_train.index.intersection(X_test.index).empty

print("Leakage checks passed.")

Scaler training sample count: 455
Expected training sample count: 455
Leakage checks passed.


In [7]:
# Small example with numeric and categorical features
sample_train = pd.DataFrame({
    "age": [22, 30, 40],
    "department": ["HR", "IT", "HR"]
})

sample_test = pd.DataFrame({
    "age": [35],
    "department": ["Finance"]  # Unseen category
})

sample_numeric = ["age"]
sample_categorical = ["department"]

sample_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, sample_numeric),
    ("categorical", categorical_pipeline, sample_categorical)
])

train_result = sample_preprocessor.fit_transform(sample_train)
test_result = sample_preprocessor.transform(sample_test)

assert train_result.shape[0] == 3
assert test_result.shape[0] == 1
assert test_result.shape[1] == train_result.shape[1]

print("Unseen-category test passed.")
print("Training shape:", train_result.shape)
print("Testing shape:", test_result.shape)

Unseen-category test passed.
Training shape: (3, 3)
Testing shape: (1, 3)
